In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Escribe tu primer programa con Qiskit Serverless

<Accordion>
<AccordionItem title="Versiones de paquetes">

El código de esta página fue desarrollado con los siguientes requisitos.
Recomendamos usar estas versiones o versiones más recientes.

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</AccordionItem>
</Accordion>
> **Tip:** **Qiskit Serverless está siendo actualizado y sus funciones están cambiando rápidamente.** Durante esta fase de desarrollo, consulta las notas de versión y la documentación más reciente en la página de [Qiskit Serverless en GitHub](https://qiskit.github.io/qiskit-serverless/index.html).

Este ejemplo demuestra cómo usar las herramientas de `qiskit-serverless` para crear un programa de transpilación en paralelo, y luego implementar `qiskit-ibm-catalog` para subir tu programa a IBM Quantum Platform y usarlo como un servicio remoto reutilizable.

## Descripción general del flujo de trabajo
1. Crea un directorio local y un archivo de programa vacío (`./source_files/transpile_remote.py`)
1. Agrega código a tu programa que, cuando se suba a Qiskit Serverless, transpilará un circuito
1. Usa `qiskit-ibm-catalog` para autenticarte en Qiskit Serverless
1. Sube el programa a Qiskit Serverless

Después de subir tu programa, puedes ejecutarlo para transpilar el circuito siguiendo la guía [Ejecuta tu primera carga de trabajo de Qiskit Serverless de forma remota](/guides/serverless-run-first-workload).

## Ejemplo: Transpilación remota con Qiskit Serverless
Este ejemplo te guía por el proceso de crear y completar un archivo de programa que, cuando lo subas a Qiskit Serverless, transpilará un `circuit` contra un `backend` dado y un `optimization_level` objetivo.

:::tip

Qiskit Serverless requiere organizar los archivos `.py` de tu carga de trabajo en un directorio dedicado. La siguiente estructura es un ejemplo de buenas prácticas:

In [ ]:
# This cell is hidden from users, it creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [ ]:
%%writefile ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

### Add code to get program arguments

Now add the following code to your program file, which sets up program arguments.

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

### Agrega código para obtener los argumentos del programa
Ahora agrega el siguiente código a tu archivo de programa, que configura los argumentos del programa.

Tu archivo inicial `transpile_remote.py` tiene tres entradas: `circuits`, `backend_name` y `optimization_level`. Serverless actualmente solo acepta entradas y salidas serializables. Por esta razón, no puedes pasar `backend` directamente, así que usa `backend_name` como cadena de texto en su lugar.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

### Agrega código que llame al backend
Agrega el siguiente código a tu archivo de programa, que llama a tu backend con `QiskitRuntimeService`.

El siguiente código asume que ya seguiste el proceso para guardar tus credenciales usando `QiskitRuntimeService.save_account`, y cargará tu cuenta guardada predeterminada a menos que especifiques lo contrario. Consulta [Guarda tus credenciales de inicio de sesión](/guides/save-credentials) e [Inicializa tu cuenta de servicio de Qiskit Runtime](/guides/initialize-account) para más información.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

# Each circuit is being transpiled and will populate the array
results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

### Agrega código para transpilar
Por último, agrega el siguiente código a tu archivo de programa. Este código ejecuta `transpile_remote()` sobre todos los `circuits` pasados como entrada y devuelve los `transpiled_circuits` como resultado:

In [13]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

<span id="upload-to-qiskit-serverless"></span>
### Autentícate en Qiskit Serverless
Usa `qiskit-ibm-catalog` para autenticarte en `QiskitServerless` con tu clave de API (puedes usar tu clave de API de `QiskitRuntimeService`, o crear una nueva clave de API en el [panel de IBM Quantum Platform](https://quantum.cloud.ibm.com)).

In [8]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [9]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

### Verify upload

To check if it successfully uploaded, use `serverless.list()`, as in the following code:

In [10]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [11]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [12]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

## Next steps

<Admonition type="info" title="Recommendations">

- Learn how to pass inputs and run your program remotely in the [Run your first Qiskit Serverless workload remotely](/docs/guides/serverless-run-first-workload) topic.

</Admonition>